In [2]:
from dotenv import load_dotenv
from llama_index.embeddings.cohere import CohereEmbedding
from llama_index.graph_stores.neo4j import Neo4jPropertyGraphStore
from llama_index.core import PropertyGraphIndex, StorageContext, Document
from neo4j import GraphDatabase
from llama_index.core import Settings
from llama_index.llms.openai_like import OpenAILike
from llama_index.llms.openrouter import OpenRouter
from openai import OpenAI
import nest_asyncio
from llama_index.graph_stores.neo4j import Neo4jPropertyGraphStore
from neo4j import GraphDatabase
from llama_index.core.graph_stores.types import EntityNode, Relation
import logging
from datetime import datetime
import unicodedata
import spacy
import unicodedata
import re
import os
import logging
import json
import hashlib
import spacy
nest_asyncio.apply()
load_dotenv(override=True)


True

In [3]:
import spacy
nlp = spacy.load("fr_core_news_sm")

In [23]:
import logging
import sys

# 1. Configuration constants
log_format = '%(asctime)s - %(levelname)s - %(message)s'
date_format = '%Y-%m-%d %H:%M:%S'
log_file = 'graph_extraction.log'

# 2. Get the root logger
root_logger = logging.getLogger()
root_logger.setLevel(logging.INFO)

# 3. Clear existing handlers (to avoid duplicates or conflicts)
if root_logger.hasHandlers():
    root_logger.handlers.clear()

# 4. Create File Handler
file_handler = logging.FileHandler(log_file, encoding='utf-8')
file_handler.setFormatter(logging.Formatter(log_format, datefmt=date_format))

# 5. Create Console Handler
console_handler = logging.StreamHandler(sys.stdout)
console_handler.setFormatter(logging.Formatter(log_format, datefmt=date_format))

# 6. Add handlers to root logger
root_logger.addHandler(file_handler)
root_logger.addHandler(console_handler)

logger = logging.getLogger(__name__)

logger.info("🚀 Logger successfully writing to both console and file!")

2026-05-01 15:13:41 - INFO - 🚀 Logger successfully writing to both console and file!


In [24]:

# 1. API Setup
os.environ["COHERE_API_KEY"] = os.getenv("COHERE_API_KEY")
OPENROUTER_API_1 = os.getenv("OPENROUTER_API_1")
OPENROUTER_API_2 = os.getenv("OPENROUTER_API_2")
OPENROUTER_API_3 = os.getenv("OPENROUTER_API_3")


ESPRIT_LLM_API_KEY = os.getenv("ESPRIT_LLM_API_KEY")
ESPRIT_MODEL_NAME =os.getenv("ESPRIT_MODEL_NAME")
ESPRIT_BASE_URL =os.getenv("ESPRIT_BASE_URL")

# 2. Neo4j Setup (AuraDB)
NEO4J_URI = os.getenv("NEO4J_URI")
NEO4J_USERNAME = "82c918a5"
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD")



# URI examples: "neo4j://localhost", "neo4j+s://xxx.databases.neo4j.io"
URI = NEO4J_URI
AUTH = (NEO4J_USERNAME, NEO4J_PASSWORD)

with GraphDatabase.driver(URI, auth=AUTH) as driver:
    driver.verify_connectivity()


# 1. Connect to Neo4j
graph_store = Neo4jPropertyGraphStore(
    username=NEO4J_USERNAME,
    password=NEO4J_PASSWORD,
    url=NEO4J_URI,
    database="82c918a5"
)




# 3. Initialize Models
# 2. Initialize LlamaIndex LLM client pointing to your server
""" llm = OpenAILike(
    model=ESPRIT_MODEL_NAME,
    api_key=ESPRIT_LLM_API_KEY,
    api_base=ESPRIT_BASE_URL, # Redirects LlamaIndex to Esprit's server
    temperature=0.1,          # Recommended for Legal Graph Extraction
    max_tokens=2048,
    is_chat_model=True,
    reuse_client=False        # Often safer with custom vLLM endpoints
)  """

OPENROUTER_API_KEYs = [OPENROUTER_API_1, OPENROUTER_API_2, OPENROUTER_API_3]
api_index = 0

llm = OpenRouter(model="openai/gpt-oss-120b:free", api_key=OPENROUTER_API_KEYs[0])

embed_model = CohereEmbedding(cohere_api_key=os.environ["COHERE_API_KEY"], model="embed-multilingual-v3.0")

Settings.llm = llm
Settings.embed_model = embed_model

In [25]:
# Load your existing JSONL file
scraped_articles = []
with open("scraped_enriched_data_truncated.jsonl", "r", encoding="utf-8") as f:
    for line in f:
        scraped_articles.append(json.loads(line))

documents = []
for art in scraped_articles:
    # Combine text and article metadata for the Graph extractor
    node_content = f"Article {art['article_number']} of {art['metadata']['source']}: {art['text']}"
    
    doc = Document(
        text=node_content,
        id_=art['id'],
        metadata={
            "article_number": art["article_number"],
            "law_source": art["metadata"]["source"],
            "keyword": art["metadata"]["keyword"]
        }
    )
    documents.append(doc)

print(f"Loaded {len(documents)} articles for Graph processing.")

Loaded 1116 articles for Graph processing.


In [26]:
from enum import Enum


class EntityLabel(str, Enum):
    # Structure de la loi
    ARTICLE = "ARTICLE"
    LAW_CODE = "LAW_CODE"       # Ex: COC, CP, Code des Droits Réels
    LEGAL_STRUCTURE = "LEGAL_STRUCTURE" # Titre, Chapitre, Section
    
    # Concepts et Actes
    LEGAL_CONCEPT = "LEGAL_CONCEPT" # Capacité, Consentement, Rescision
    LEGAL_ACT = "LEGAL_ACT"         # Contrat, Donation, Mariage, Délit
    
    # Sujets et Objets
    PERSON_TYPE = "PERSON_TYPE"     # Mineur, Créancier, Débiteur, Tuteur
    LEGAL_ENTITY = "LEGAL_ENTITY"   # Société, État, Association
    ASSET = "ASSET"                 # Immeuble, Meuble, Gage
    
    # Paramètres de la règle
    CONDITION = "CONDITION"         # Un évènement ou pré-requis
    PENALTY = "PENALTY"             # Amende, Emprisonnement, Dommages-intérêts
    DEADLINE = "DEADLINE"           # Prescription, délais de recours
    LEGAL_CONSEQUENCE = "LEGAL_CONSEQUENCE" # Nullité, Extinction, Transfert de propriété
    
    # Autorités
    AUTHORITY = "AUTHORITY"         # Tribunal, Huissier, Conservateur

class RelationLabel(str, Enum):
    # Hiérarchie et Référence
    CITES = "CITES"                 # Référence directe à un article
    PART_OF = "PART_OF"             # Article appartient à un Code ou Chapitre
    MODIFIES = "MODIFIES"           # Un article qui en amende un autre
    REPLACES = "REPLACES"           # Abrogation
    
    # Définition et Logique
    DEFINES = "DEFINES"             # L'article explique un concept
    ESTABLISHES = "ESTABLISHES"     # Crée une règle ou une présomption
    GOVERNS = "GOVERNS"             # L'article régit une situation
    
    # Conditions et Déclencheurs
    REQUIRES = "REQUIRES"           # Condition Sine Qua Non
    TRIGGERS = "TRIGGERS"           # Si A alors B se déclenche
    ENABLES = "ENABLES"             # Donne le droit de
    ENABLED_BY = "ENABLED_BY"           # Un droit qui dépend d'une condition
    CONDITIONAL_ON = "CONDITIONAL_ON"
    
    # Contraintes et Exceptions
    PROHIBITS = "PROHIBITS"         # Interdiction formelle
    EXCEPT_TO = "EXCEPT_TO"         # Dérogation (Lex Specialis)
    LIMITS = "LIMITS"               # Restriction d'un droit existant
    LIMITED_BY = "LIMITED_BY"           # Un droit limité par une condition
    EXEMPTS = "EXEMPTS"             # Dispense d'une obligation
    
    # Effets Juridiques
    EXTINGUISHES = "EXTINGUISHES"   # Fin d'une obligation (paiement, prescription)
    NULLIFIES = "NULLIFIES"         # Entraîne la nullité
    OBLIGATES = "OBLIGATES"         # Crée une dette ou un devoir
    TRANSFERS = "TRANSFERS"         # Transfert de droit ou propriété
    
    # Application
    APPLIES_TO = "APPLIES_TO"       # Champ d'application (personnes ou biens)
    PUNISHES = "PUNISHES"           # Lie une infraction à une peine
    
# 2. Extract allowed values
allowed_entities = [e.value for e in EntityLabel]
allowed_relations = [r.value for r in RelationLabel]

In [27]:
system_prompt_template = """
### RÔLE ###
Tu es un ingénieur expert en modélisation de graphes de connaissances juridiques. 
Ton but est de transformer le texte brut en un réseau logique de concepts interconnectés.

### DIRECTIVES DE MODÉLISATION ###
1. 1. **ZÉRO GÉNÉRIQUE :** Les entités 'Loi', 'Droit', 'Code', 'Justice' ou 'Système' sont INTERDITES. Utilise les noms précis (ex: 'COC', 'Article 2').
2. **HIÉRARCHIE ET SOURCE :** Relie toujours l'Article à son Code source (LAW_CODE).
3. **GRANULARITÉ :** Si un texte énumère des conditions (ex: "Les éléments sont : A, B et C"), crée des triplets séparés pour chaque élément.
4. **ATOMISATION :** Chaque entité doit être un concept court (1-3 mots). Pas de phrases.

### SCHÉMA AUTORISÉ ###
- Labels : {entities}
- Relations : {relations}

### EXEMPLE DE RÉFÉRENCE (Basé sur l'Article 2) ###
Texte : "Article 2 du COC : Les éléments nécessaires pour la validité des obligations sont : la capacité de s'obliger, un objet certain et une cause licite."
Sortie attendue : [
  {{"entity1": "Article 2", "entity1_label": "ARTICLE", "relation": "CITES", "entity2": "COC", "entity2_label": "LAW_CODE"}},
  {{"entity1": "Article 2", "entity1_label": "ARTICLE", "relation": "DEFINES", "entity2": "Validité des obligations", "entity2_label": "LEGAL_CONCEPT"}},
  {{"entity1": "Validité des obligations", "entity1_label": "LEGAL_CONCEPT", "relation": "REQUIRES", "entity2": "Capacité de s'obliger", "entity2_label": "CONDITION"}},
  {{"entity1": "Validité des obligations", "entity1_label": "LEGAL_CONCEPT", "relation": "REQUIRES", "entity2": "Objet certain", "entity2_label": "CONDITION"}},
  {{"entity1": "Validité des obligations", "entity1_label": "LEGAL_CONCEPT", "relation": "REQUIRES", "entity2": "Cause licite", "entity2_label": "CONDITION"}}
]

### TEXTE À ANALYSER ###
{text}

### SORTIE JSON EXHAUSTIVE ET PRÉCISE (STRICTEMENT AUCUN TEXTE AVANT OU APRÈS) ###
"""

In [8]:
import json

# 1. Take sample
test_article = documents[9].text

try:
    print("🤖 Sending to LLM...")
    response = llm.complete(system_prompt_template.format(
        entities=allowed_entities,
        relations=allowed_relations,
        text=test_article
    ))
    raw_text = response.text
    
    # 4. Parsing the List of Dicts
    start = raw_text.find('[')
    end = raw_text.rfind(']') + 1
    extracted_dicts = json.loads(raw_text[start:end])
    
    valid_records = []
    invalid_records = []

    for entry in extracted_dicts:
        # Check if all labels and relations match your Pydantic Enums
        rel_ok = entry.get('relation') in allowed_relations
        ent1_ok = entry.get('entity1_label') in allowed_entities
        ent2_ok = entry.get('entity2_label') in allowed_entities
        
        if rel_ok and ent1_ok and ent2_ok:
            valid_records.append(entry)
        else:
            invalid_records.append(entry)

    # 5. The "Matched" Output
    print("\n--- ✅ VALIDATED RECORDS (Strict Pydantic Compliance) ---")
    for r in valid_records:
        # We print it like a triplet for readability, but it's coming from the Dict
        print(f"   MATCHED: ({r['entity1_label']}) {r['entity1']} --[{r['relation']}]--> ({r['entity2_label']}) {r['entity2']}")

    if invalid_records:
        print("\n--- ⚠️ INVALID RECORDS (Enum Mismatch) ---")
        for r in invalid_records:
            print(f"   REJECTED: {r}")

except Exception as e:
    print(f"❌ Error: {e}")

🤖 Sending to LLM...


2026-05-01 15:05:42 - INFO - HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"



--- ✅ VALIDATED RECORDS (Strict Pydantic Compliance) ---
   MATCHED: (ARTICLE) Article 10 --[CITES]--> (LAW_CODE) Code Obligations Contrats
   MATCHED: (ARTICLE) Article 10 --[ENABLED_BY]--> (LEGAL_CONCEPT) Attaque de l'obligation
   MATCHED: (LEGAL_CONCEPT) Attaque de l'obligation --[APPLIES_TO]--> (PERSON_TYPE) tuteur
   MATCHED: (LEGAL_CONCEPT) Attaque de l'obligation --[APPLIES_TO]--> (PERSON_TYPE) mineur après majorité
   MATCHED: (PERSON_TYPE) mineur après majorité --[CONDITIONAL_ON]--> (CONDITION) Manœuvres frauduleuses
   MATCHED: (CONDITION) Manœuvres frauduleuses --[PROHIBITS]--> (LEGAL_CONCEPT) Attaque de l'obligation
   MATCHED: (CONDITION) Manœuvres frauduleuses --[ENABLED_BY]--> (LEGAL_CONCEPT) Autorisation du tuteur
   MATCHED: (CONDITION) Manœuvres frauduleuses --[ENABLED_BY]--> (LEGAL_CONCEPT) Qualité de commerçant
   MATCHED: (PERSON_TYPE) Mineur --[OBLIGATES]--> (ASSET) Profit retiré de l'obligation
   MATCHED: (ASSET) Profit retiré de l'obligation --[LIMITED_BY]-->

In [28]:
def generate_text_hash(text):
    # Crée un ID unique basé sur le contenu exact du texte
    return hashlib.md5(text.encode('utf-8')).hexdigest()

In [29]:

def normalize_id(text):
    if not text:
        return "unknown"

    # 1. Nettoyage de base
    text = text.lower().strip()
    
    # 2. Analyse avec spaCy
    doc = nlp(text)
    
    # 3. FILTRAGE STRATÉGIQUE :
    # On ne garde que les Noms (NOUN), les Noms Propres (PROPN), 
    # les Adjectifs (ADJ) et les Nombres (NUM).
    # On ignore les articles (DET), les prépositions (ADP), etc.
    lemmas = []
    for token in doc:
        if token.pos_ in ["NOUN", "ADJ", "PROPN", "NUM", "VERB"] or token.text.isdigit():
            # On récupère la racine (lemme)
            # Optionnel : Ignorer les verbes trop communs (être, avoir)
            if token.lemma_ not in ["etre", "avoir"]:
                lemmas.append(token.lemma_)
    
    # Si le filtrage est trop strict (vide), on garde le texte original
    if not lemmas:
        lemmas = [t.text for t in doc if not t.is_stop]
        
    # 4. Reconstruction de l'ID
    normalized_text = "_".join(lemmas)

    # 5. Suppression des accents
    nfkd = unicodedata.normalize('NFKD', normalized_text)
    normalized = ''.join([c for c in nfkd if not unicodedata.combining(c)])
    
    # 6. Nettoyage final des caractères spéciaux
    normalized = normalized.replace(" ", "_").replace("-", "_").replace("'", "_")
    normalized = ''.join(c for c in normalized if c.isalnum() or c == '_')
    
    # Nettoyage des underscores doubles
    normalized = re.sub(r'_+', '_', normalized).strip('_')
    
    return normalized

# ============ ZONE DE TEST AMÉLIORÉE ============
test_cases = [
    "Les Obligations", 
    "L'obligation", 
    "Contractuelles", 
    "Déclaration d'incapacité",
    "Article 3",
    "La capacité de s'obliger"
]

print("🚀 Test de normalisation PRO (Filtrage POS + Lemmatisation):")
for w in test_cases:
    print(f"'{w}' -> {normalize_id(w)}")

🚀 Test de normalisation PRO (Filtrage POS + Lemmatisation):
'Les Obligations' -> obligation
'L'obligation' -> obligation
'Contractuelles' -> contractuel
'Déclaration d'incapacité' -> declaration_incapacite
'Article 3' -> article_3
'La capacité de s'obliger' -> capacite_obliger


In [36]:

# ============ BATCH PROCESSING SETUP ============
BATCH_SIZE = 5
batch_nodes_dict = {}      # Dict pour déduplication: {id -> EntityNode}
batch_relations = []       # Liste plate pour les relations
total_nodes = 0
total_relations = 0
total_errors = 0
total_deduplicated = 0
start_time = datetime.now()

batch_num = 2
START_INDEX = (batch_num-1) * BATCH_SIZE
current_batch_num = START_INDEX // BATCH_SIZE


BACKUP_FILE_PATH = "law_graph_backup.jsonl"
keys_exhausted = False




In [37]:

from time import sleep
storage_context = StorageContext.from_defaults(graph_store=graph_store)
logger.info(f"🚀 Starting Professional Manual Migration for {len(documents)} articles in batches of {BATCH_SIZE}...")

with open(BACKUP_FILE_PATH, "a", encoding="utf-8", buffering=1) as backup_file:
    try : 

        for idx, doc in enumerate(documents[START_INDEX:], start=START_INDEX + 1):
            art_num = doc.metadata['article_number']
            law_source = doc.metadata['law_source']
            
            # Générer l'ID unique de l'ARTICLE
            node_content = f"Article {art_num} of {law_source}: {doc.text}"
            article_hash_id = generate_text_hash(node_content)
            
            logger.info(f"[{idx}/{len(documents)}] Processing Article {art_num} from {law_source}...")
            
            try:
                # Appeler l'LLM pour extraire les triplets
                system_prompt = system_prompt_template.format(
                    entities=", ".join(allowed_entities),
                    relations=", ".join(allowed_relations),
                    text=doc.text
                )
                # Set a maximum number of retries
                max_retries = len(OPENROUTER_API_KEYs)
                attempts = 0
                raw_text = None
                
                while attempts <= max_retries: # <= pour inclure la tentative sur le serveur ESPRIT
                    try:
                        response = llm.complete(system_prompt)
                        raw_text = response.text
                        break # Succès ! On sort du while
                        
                    except Exception as llm_err:
                        attempts += 1
                        err_msg = str(llm_err).lower()
                        
                        # CAS 1 : On a encore des clés OpenRouter à tester
                        if attempts < max_retries:
                            api_index = (api_index + 1) % len(OPENROUTER_API_KEYs)
                            logger.warning(f"   ⚠ OpenRouter Key {api_index-1} failed. Rotating to key {api_index}...")
                            llm = OpenRouter(model="openai/gpt-oss-120b:free", api_key=OPENROUTER_API_KEYs[api_index])
                            sleep(2)
                            continue
                        
                        # CAS 2 : TOUTES les clés OpenRouter ont échoué -> Basculement sur ESPRIT
                        elif attempts == max_retries:
                            logger.error("   🚨 ALL OpenRouter keys failed. Reverting to ESPRIT Hosted LLM...")
                            llm = OpenAILike(
                                model=ESPRIT_MODEL_NAME,
                                api_key=ESPRIT_LLM_API_KEY,
                                api_base=ESPRIT_BASE_URL,
                                temperature=0.1,
                                max_tokens=2048,
                                is_chat_model=True,
                                timeout=600 # On donne du temps au serveur local
                            )
                            continue # On retente une dernière fois avec 'llm' qui est maintenant ESPRIT
                        
                        # CAS 3 : Même le serveur ESPRIT a échoué
                        else:
                            logger.critical(f"   💀 Fatal: Both OpenRouter and ESPRIT failed for Art {art_num}")
                            keys_exhausted = True
                            break
                        
                if keys_exhausted:
                    break 

                # 2. Empty Response: Skip this article but keep going to the next
                if not raw_text or raw_text.strip() == "":
                    logger.warning(f"Article {art_num} returned empty text. Skipping to next.")
                    total_errors += 1
                    continue
                
                # Parse JSON robuste
                start, end = raw_text.find('['), raw_text.rfind(']') + 1

                if start == -1 or end <= start:
                    logger.warning(f"   ⚠ No JSON found in LLM response")
                    continue
                    
                extracted_data = json.loads(raw_text[start:end])
                
                logger.debug(f"   ✓ LLM extracted {len(extracted_data)} relations")

                # --- 💾 SAUVEGARDE BACKUP (JSONL) ---
                backup_entry = {
                    "idx": idx,
                    "timestamp": datetime.now().isoformat(),
                    "article_metadata": {
                        "number": art_num,
                        "source": law_source,
                        "hash": article_hash_id
                    },
                    "graph_records": extracted_data # Contient les 5 clés par entrée
                }
                backup_file.write(json.dumps(backup_entry, ensure_ascii=False) + "\n")


                relations_local = []

                # Ajouter le nœud de l'Article lui-même
                if article_hash_id not in batch_nodes_dict:
                    main_art_node = EntityNode(
                        id=article_hash_id, 
                        name=f"Article {art_num}", 
                        label="ARTICLE",
                        properties={
                            "article_number": art_num,
                            "text": doc.text, 
                            "source": law_source}
                    )
                    batch_nodes_dict[article_hash_id] = main_art_node
                else:
                    total_deduplicated += 1

                for entry in extracted_data:
                    # Logique d'ID intelligente
                    if entry['entity1'].lower() in [f"article {art_num}".lower(), str(art_num)]:
                        id1 = article_hash_id
                    else:
                        id1 = normalize_id(entry['entity1'])
                        
                    if entry['entity2'].lower() in [f"article {art_num}".lower(), str(art_num)]:
                        id2 = article_hash_id
                    else:
                        id2 = normalize_id(entry['entity2'])

                    # Ajouter les nœuds au dict (déduplication automatique par clé)
                    if id1 not in batch_nodes_dict:
                        source_node = EntityNode(id=id1, name=entry['entity1'], label=entry['entity1_label'])
                        batch_nodes_dict[id1] = source_node
                    else:
                        total_deduplicated += 1
                        
                    if id2 not in batch_nodes_dict:
                        target_node = EntityNode(id=id2, name=entry['entity2'], label=entry['entity2_label'])
                        batch_nodes_dict[id2] = target_node
                    else:
                        total_deduplicated += 1
                    
                    # Créer la relation
                    rel = Relation(
                        label=entry['relation'],
                        source_id=id1,
                        target_id=id2
                    )
                    relations_local.append(rel)
                
                batch_relations.extend(relations_local)
                
                logger.info(f"   ✓ Article {art_num}: {len(relations_local)} relations (Dict size: {len(batch_nodes_dict)} unique nodes)")
                
                # ============ FLUSH BATCH EVERY 30 ARTICLES ============
                if idx % BATCH_SIZE == 0 or idx == len(documents):
                    logger.info(f"\n{'='*60}")
                    logger.info(f"📤 FLUSHING BATCH at document {idx}/{len(documents)}...")
                    logger.info(f"   Unique nodes to upsert: {len(batch_nodes_dict)}")
                    logger.info(f"   Relations to upsert: {len(batch_relations)}")
                    logger.info(f"   Deduplicated (avoided): {total_deduplicated}")
                    
                    if batch_nodes_dict:
                        try:
                            # Convertir dict.values() en liste
                            nodes_list = list(batch_nodes_dict.values())
                            graph_store.upsert_nodes(nodes_list)
                            graph_store.upsert_relations(batch_relations)
                            total_nodes += len(nodes_list)
                            total_relations += len(batch_relations)
                            batch_num += 1
                            logger.info(f"   ✓ Batch {batch_num} committed successfully!")
                            logger.info(f"   📊 Cumulative: {total_nodes} nodes, {total_relations} relations")
                        except Exception as upsert_err:
                            logger.error(f"   ✗ Error upserting batch: {upsert_err}")
                            total_errors += 1    
                    # Reset batch
                    batch_nodes_dict = {}
                    batch_relations = []
                    logger.info(f"{'='*60}\n")
                sleep(2)  # Small delay to avoid hitting rate limits too quickly 2s
            except Exception as e:
                logger.error(f"   ✗ Error processing Article {art_num}: {e}")
                total_errors += 1


        if batch_nodes_dict or batch_relations:
            logger.info(f"📤 Final flush for remaining {len(batch_nodes_dict)} nodes...")
            try:
                graph_store.upsert_nodes(list(batch_nodes_dict.values()))
                graph_store.upsert_relations(batch_relations)
                total_nodes += len(batch_nodes_dict)
                total_relations += len(batch_relations)
            except Exception as e:
                logger.error(f"   ✗ Final flush error: {e}")

        # ============ FINAL SUMMARY ============
        end_time = datetime.now()
        duration = (end_time - start_time).total_seconds()

        logger.info(f"\n{'='*60}")
        logger.info(f"🏆 MIGRATION COMPLETE")
        logger.info(f"{'='*60}")
        logger.info(f"📊 FINAL STATISTICS:")
        logger.info(f"   Total documents processed: {len(documents)}")
        logger.info(f"   Total nodes created: {total_nodes}")
        logger.info(f"   Total relations created: {total_relations}")
        logger.info(f"   Total errors: {total_errors}")
        logger.info(f"   Duplicates avoided: {total_deduplicated}")
        logger.info(f"   Duration: {duration:.2f} seconds")
        if duration > 0:
            logger.info(f"   Average: {len(documents)/duration:.2f} articles/sec")
        logger.info(f"{'='*60}")
    
    except KeyboardInterrupt:
        logger.warning("🛑 Script interrupted by user. Closing backup file properly.")
    except Exception as fatal_e:
        logger.error(f"🔥 FATAL ERROR: {fatal_e}")
    finally:
        # Le fichier est automatiquement fermé ici grâce au bloc 'with'
        logger.info(f"🔒 Backup file {BACKUP_FILE_PATH} has been closed.")

logger.info("🏆 Process finished.")


2026-05-01 15:27:30 - INFO - 🚀 Starting Professional Manual Migration for 1116 articles in batches of 5...
2026-05-01 15:27:30 - INFO - [6/1116] Processing Article 6 from Code Obligations Contrats...


2026-05-01 15:27:31 - INFO - HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"
2026-05-01 15:27:42 - INFO -    ✓ Article 6: 6 relations (Dict size: 7 unique nodes)
2026-05-01 15:27:44 - INFO - [7/1116] Processing Article 7 from Code Obligations Contrats...
2026-05-01 15:27:45 - INFO - HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"
2026-05-01 15:28:03 - INFO -    ✓ Article 7: 5 relations (Dict size: 12 unique nodes)
2026-05-01 15:28:05 - INFO - [8/1116] Processing Article 8 from Code Obligations Contrats...
2026-05-01 15:28:07 - INFO - HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"
2026-05-01 15:28:42 - INFO -    ✓ Article 8: 10 relations (Dict size: 22 unique nodes)
2026-05-01 15:28:44 - INFO - [9/1116] Processing Article 9 from Code Obligations Contrats...
2026-05-01 15:28:45 - INFO - HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"
2026-05-01 15:2

In [28]:
print("🔍 Querying Neo4j for nodes...")
verify_driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USERNAME, NEO4J_PASSWORD))
with verify_driver.session(database="82c918a5") as session:
    result = session.run("MATCH (n) RETURN count(n) AS node_count")
    count = result.single()["node_count"]
    print(f"📊 Final count in Neo4j: {count} nodes")

verify_driver.close()

if count > 0:
    print("🏆 SUCCESS! Data is now visible in your LawGraph.")
else:
    print("❌ Still 0? Check your OpenRouter dashboard to see if the requests were fulfilled.")

🔍 Querying Neo4j for nodes...
📊 Final count in Neo4j: 28 nodes
🏆 SUCCESS! Data is now visible in your LawGraph.


In [29]:
# Define the query engine
query_engine = index.as_query_engine(
    include_text=True, # Include raw text in the response
    similarity_top_k=5 # Vector search depth
)

query = "Quelles sont les conséquences de la lésion pour un mineur ?"
response = query_engine.query(query)

print(f"⚖️ Réponse GraphRAG:\n{response}")

NameError: name 'index' is not defined